<a href="https://colab.research.google.com/github/ipavlopoulos/diachronic-greek-letterforms/blob/main/notebooks/resnet18_pt_ft_standard_scl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResNet18-PT+FT With Standard SCL

This notebook trains a pretrained ResNet-18 on Hell-Char and fine-tunes all layers using cross-entropy plus the standard supervised contrastive loss from Khosla et al. (2020).

Configuration:

- `PT+FT`: ImageNet-pretrained ResNet-18, fine-tuned end to end.
- `SCL`: standard supervised contrastive loss with same-class positives.
- No `LF`: the lacuna-driven erasure transform is not used.
- No `DSCL`: class-similarity-weighted negatives are not used.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import random
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/ipavlopoulos/diachronic-greek-letterforms.git"
REPO_DIR = Path("/content/diachronic-greek-letterforms")

if IN_COLAB and not Path("source.py").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
elif not Path("source.py").exists() and Path("..").joinpath("source.py").exists():
    os.chdir("..")
elif not Path("source.py").exists() and Path("source").joinpath("source.py").exists():
    os.chdir("source")

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

from source import ImageDatasetAugmented, ResNetClassifier, SupConLoss

print(f"Working directory: {ROOT}")

In [ ]:
SEED = 42
BATCH_SIZE = 16
NUM_EPOCHS = 100
LEARNING_RATE = 1e-4
LAMBDA_SCL = 0.1
N_VIEWS = 2
PATIENCE = 10
TEST_SIZE = 0.2
VAL_SIZE = 0.1
NUM_WORKERS = 2
OUTPUT_DIR = ROOT / "runs" / "resnet18_pt_ft_standard_scl"
CHECKPOINT_PATH = OUTPUT_DIR / "best_resnet18_pt_ft_standard_scl.pth"
SUMMARY_PATH = OUTPUT_DIR / "resnet18_pt_ft_standard_scl_summary.json"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {device}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Load Hell-Char

Labels are inferred from the released Hell-Char cliplet filenames. Unknown labels are removed before splitting.

In [ ]:
def preprocess_image_2d(image_path, size=(64, 64), otsu=False):
    img = Image.open(image_path).convert("L")
    img_np = np.array(img)
    if otsu:
        _, img_np = cv2.threshold(img_np, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        img_np = 255 - img_np
    img_resized = cv2.resize(img_np, size, interpolation=cv2.INTER_AREA)
    return img_resized.astype(np.float32) / 255.0

cliplet_dir = ROOT / "data" / "hellchar" / "cliplets"
image_paths = sorted(p for p in cliplet_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
data = pd.DataFrame({"path": image_paths, "filename": [p.name for p in image_paths]})
data["letter"] = data.filename.apply(lambda name: name.split("_")[0])
data = data[data["letter"] != "Unknown"].copy().reset_index(drop=True)

image_data = np.array([preprocess_image_2d(path) for path in data.path])
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(data["letter"])
num_classes = len(label_encoder.classes_)

print(f"Images: {len(data)}")
print(f"Classes: {num_classes}")
print(label_encoder.classes_.tolist())

## Split And Dataloaders

The training transform uses standard image augmentations only. It deliberately excludes `RandomLacunae`, so this is not an LF run.

In [ ]:
indices = np.arange(len(data))
train_indices, test_indices, y_train, y_test = train_test_split(
    indices,
    labels,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=labels,
)
train_indices, val_indices, y_train, y_val = train_test_split(
    train_indices,
    y_train,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y_train,
)

train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomResizedCrop(size=(64, 64), scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_loader = DataLoader(
    ImageDatasetAugmented(image_data[train_indices], y_train, transform=train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    ImageDatasetAugmented(image_data[val_indices], y_val, transform=eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    ImageDatasetAugmented(image_data[test_indices], y_test, transform=eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train/val/test: {len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}")

## Train

The objective is:

```text
loss = cross_entropy + lambda_scl * standard_supcon
```

Unlike DSCL, the contrastive denominator is not reweighted by a class-similarity matrix. To keep this baseline comparable with the LF+DSCL ResNet run, the SCL term is computed on two standard augmented views per training image. The LF lacuna transform remains disabled.

In [ ]:
standard_scl_view_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomResizedCrop(size=(64, 64), scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])


def get_standard_scl_embeddings(model, batch, labels_batch, n_views, view_transform, device):
    augmented = []
    for image in batch:
        image_cpu = image.detach().cpu()
        for _ in range(n_views):
            augmented.append(view_transform(image_cpu))
    augmented = torch.stack(augmented).to(device)
    augmented_labels = labels_batch.repeat_interleave(n_views)
    embeddings = model.get_embeddings(augmented)
    return embeddings, augmented_labels


def train_resnet_standard_scl(model, train_loader, val_loader, device):
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=3,
        factor=0.1,
    )
    ce_loss_fn = nn.CrossEntropyLoss()
    scl_loss_fn = SupConLoss(temperature=0.07)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_accuracy": []}

    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0.0
        for inputs, labels_batch in train_loader:
            inputs = inputs.to(device)
            labels_batch = labels_batch.to(device)

            logits = model(inputs)
            embeddings, scl_labels = get_standard_scl_embeddings(
                model,
                inputs,
                labels_batch,
                N_VIEWS,
                standard_scl_view_transform,
                device,
            )
            loss = ce_loss_fn(logits, labels_batch) + LAMBDA_SCL * scl_loss_fn(embeddings, scl_labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels_batch in val_loader:
                inputs = inputs.to(device)
                labels_batch = labels_batch.to(device)
                logits = model(inputs)
                loss = ce_loss_fn(logits, labels_batch)
                val_loss += loss.item() * inputs.size(0)
                predicted = logits.argmax(dim=1)
                total += labels_batch.size(0)
                correct += (predicted == labels_batch).sum().item()

        val_loss /= len(val_loader.dataset)
        val_accuracy = correct / total
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        print(
            f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
            f"Train Loss: {train_loss:.4f}, "
            f"Val Loss: {val_loss:.4f}, "
            f"Val Accuracy: {val_accuracy:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), CHECKPOINT_PATH)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping triggered after {epoch + 1} epochs.")
                break

    return history

model = ResNetClassifier(num_classes=num_classes, pretrained=True).to(device)
history = train_resnet_standard_scl(model, train_loader, val_loader, device)

## Evaluate

In [ ]:
def collect_predictions(model, loader, device):
    model.eval()
    pred, gold = [], []
    with torch.no_grad():
        for inputs, labels_batch in loader:
            inputs = inputs.to(device)
            logits = model(inputs)
            pred.extend(logits.argmax(dim=1).cpu().numpy())
            gold.extend(labels_batch.numpy())
    return np.array(gold), np.array(pred)

best_model = ResNetClassifier(num_classes=num_classes, pretrained=False).to(device)
best_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
y_true, y_pred = collect_predictions(best_model, test_loader, device)

true_labels = label_encoder.inverse_transform(y_true)
pred_labels = label_encoder.inverse_transform(y_pred)
report = classification_report(true_labels, pred_labels, zero_division=0, output_dict=True)
print(classification_report(true_labels, pred_labels, zero_division=0))
print(json.dumps({
    "test_accuracy": report["accuracy"],
    "macro_f1": report["macro avg"]["f1-score"],
    "weighted_f1": report["weighted avg"]["f1-score"],
}, indent=2))

## Save Summary

In [ ]:
summary = {
    "checkpoint": str(CHECKPOINT_PATH),
    "classes": label_encoder.classes_.tolist(),
    "train_size": len(train_loader.dataset),
    "val_size": len(val_loader.dataset),
    "test_size": len(test_loader.dataset),
    "history": history,
    "test_report": report,
    "settings": {
        "model": "ResNet18-PT+FT",
        "pretrained": True,
        "fine_tuned_layers": "all",
        "loss": "cross_entropy + lambda_scl * standard_supcon",
        "lambda_scl": LAMBDA_SCL,
        "n_views": N_VIEWS,
        "lf_lacuna_augmentation": False,
        "dscl_similarity_weighting": False,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
    },
}

with open(SUMMARY_PATH, "w") as handle:
    json.dump(summary, handle, indent=2, default=str)

print(f"Saved checkpoint: {CHECKPOINT_PATH}")
print(f"Saved summary: {SUMMARY_PATH}")